# ShieldDefence: entrenamiento y despliegue

Este notebook entrena los dos modelos de ShieldDefence y despliega la aplicación en Hugging Face Spaces.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4***.
2. Crea un token en https://huggingface.co/settings/tokens con permiso de **escritura** (*Write*).
3. En Colab, abre la llave 🔑 del panel izquierdo, agrega un secreto llamado `HF_TOKEN` con ese token y activa el acceso del notebook.
Después solo hay que ejecutar las celdas en orden. Todo junto toma unos 45 minutos.

## 1. Configuración

In [ ]:
HF_USER = "juandaGomez"                         # tu usuario de Hugging Face
SPACE = f"{HF_USER}/shielddefence"              # donde queda la aplicación
LLM_REPO = f"{HF_USER}/shielddefence-explicador"  # donde queda el LLM ajustado
BASE_LLM = "Qwen/Qwen2.5-0.5B-Instruct"         # modelo base para LoRA

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!nvidia-smi -L

## 2. Descargar el proyecto desde GitHub

In [ ]:
%cd /content
!rm -rf ShieldDefence && git clone -q https://github.com/JuandaGomez12/ShieldDefence
%cd /content/ShieldDefence
!ls

## 3. Instalar librerías y descargar los datos

In [ ]:
!pip install -q -r requirements-train.txt
!bash training/download_data.sh

## 4. Armar el dataset balanceado

In [ ]:
!python training/prepare_data.py

## 5. Traducir la mitad de los correos al español
Usa MarianMT (`Helsinki-NLP/opus-mt-en-es`). Cada correo traducido queda en la misma partición que su original.

In [ ]:
!python training/translate_es.py --frac 0.5

## 6. Modelo base: TF-IDF + regresión logística

In [ ]:
!python training/train_baseline.py --data data/processed_bilingue

## 7. Entrenar la BiLSTM y exportarla a NumPy

In [ ]:
!python training/train_lstm.py --data data/processed_bilingue --epochs 10
!python training/export_lstm.py --data data/processed_bilingue

In [ ]:
import json
r = json.load(open("models/lstm_config.json"))
print("Umbral:", r["umbral"], "| épocas:", r["epocas"])
print("Prueba:", {k: r["test"][k] for k in ("accuracy", "precision", "recall", "f1", "matriz")})
print("Por fuente:", r["test"]["por_fuente"])
print("Español (24 correos):", {k: r["spanish"][k] for k in ("accuracy", "precision", "recall", "f1", "matriz")})

## 8. Ejemplos de explicación y ajuste fino del LLM con LoRA

In [ ]:
!python training/make_explanations.py --data data/processed_bilingue --n 3000
!python training/train_lora.py --base {BASE_LLM} --data data/explanations --out models/explicador --epochs 2 --push_to {LLM_REPO}

## 9. Evaluar el LLM

In [ ]:
!python training/eval_llm.py --model models/explicador/fusionado

## 10. Desplegar en Hugging Face Spaces

In [ ]:
!python deploy/push_space.py --space {SPACE} --llm {LLM_REPO}
print(f"La aplicación queda en https://huggingface.co/spaces/{SPACE}")

## 11. Descargar resultados
El zip trae las métricas y los modelos de la LSTM para el documento.

In [ ]:
!zip -q -j resultados_shielddefence.zip models/*.json models/lstm_pesos.npz models/lstm_vocab.json models/explicador/lora_info.json
from google.colab import files
files.download("resultados_shielddefence.zip")